-We will work with "https://www.kaggle.com/datasets/vijayvvenkitesh/microsoft-stock-time-series-analysis " that analyse the stock price of microsoft over 6 years period.

-the model used will be the **ARIMA** model



In [1]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import os
# Download latest version
path = kagglehub.dataset_download("vijayvvenkitesh/microsoft-stock-time-series-analysis")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/microsoft-stock-time-series-analysis


In [2]:
file_path = os.path.join(path, 'Microsoft_Stock.csv')
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'])
df

,Date,Open,High,Low,Close,Volume
0,2015-04-01 16:00:00,40.60,40.76,40.31,40.72,36865322
1,2015-04-02 16:00:00,40.66,40.74,40.12,40.29,37487476
2,2015-04-06 16:00:00,40.34,41.78,40.18,41.55,39223692
3,2015-04-07 16:00:00,41.61,41.91,41.31,41.53,28809375
4,2015-04-08 16:00:00,41.48,41.69,41.04,41.42,24753438
...,...,...,...,...,...,...
1506,2021-03-25 16:00:00,235.30,236.94,231.57,232.34,34061853
1507,2021-03-26 16:00:00,231.55,236.71,231.55,236.48,25479853
1508,2021-03-29 16:00:00,236.59,236.80,231.88,235.24,25227455
1509,2021-03-30 16:00:00,233.53,233.85,231.10,231.85,24792012


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1511 entries, 0 to 1510
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    1511 non-null   datetime64[ns]
 1   Open    1511 non-null   float64       
 2   High    1511 non-null   float64       
 3   Low     1511 non-null   float64       
 4   Close   1511 non-null   float64       
 5   Volume  1511 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1)
memory usage: 71.0 KB


In [4]:
df.describe()

,Date,Open,High,Low,Close,Volume
count,1511,1511.000000,1511.000000,1511.000000,1511.000000,1.511000e+03
mean,2018-03-31 17:23:44.751820032,107.385976,108.437472,106.294533,107.422091,3.019863e+07
min,2015-04-01 16:00:00,40.340000,40.740000,39.720000,40.290000,1.016120e+05
25%,2016-09-29 04:00:00,57.860000,58.060000,57.420000,57.855000,2.136213e+07
50%,2018-04-02 16:00:00,93.990000,95.100000,92.920000,93.860000,2.662962e+07
75%,2019-10-01 04:00:00,139.440000,140.325000,137.825000,138.965000,3.431962e+07
max,2021-03-31 16:00:00,245.030000,246.130000,242.920000,244.990000,1.352271e+08
std,NaN,56.691333,57.382276,55.977155,56.702299,1.425266e+07


In [5]:
import plotly.graph_objects as go

features_to_plot = ['Open', 'High', 'Low', 'Close']

fig = go.Figure()

for feature in features_to_plot:
    fig.add_trace(go.Scatter(x=df['Date'], y=df[feature], mode='lines', name=feature))

fig.update_layout(
    title='Progress of Stock Features Over Time',
    xaxis_title='Date',
    yaxis_title='Value',
    xaxis_tickangle=-45,
    legend_title='Features',
    hovermode="x unified"
)

fig.show()

In [6]:
features_to_plot = ['Volume']

fig = go.Figure()

for feature in features_to_plot:
    fig.add_trace(go.Scatter(x=df['Date'], y=df[feature], mode='lines', name=feature))

fig.update_layout(
    title='Progress of Stock Volume Over Time',
    xaxis_title='Date',
    yaxis_title='Value',
    xaxis_tickangle=-45,
    legend_title='Features',
    hovermode="x unified"
)

fig.show()

to decrease the model overfitting and ensure a synthetic generation we will only sample data each second day at 16h. Any observation taken at different time will not be taken unto account.

In [7]:
df = df.set_index('Date')
df = df.asfreq('2D')
df

,Open,High,Low,Close,Volume
Date,,,,,
2015-04-01 16:00:00,40.60,40.76,40.31,40.72,36865322.0
2015-04-03 16:00:00,NaN,NaN,NaN,NaN,NaN
2015-04-05 16:00:00,NaN,NaN,NaN,NaN,NaN
2015-04-07 16:00:00,41.61,41.91,41.31,41.53,28809375.0
2015-04-09 16:00:00,41.25,41.62,41.25,41.48,25723861.0
...,...,...,...,...,...
2021-03-22 16:00:00,230.27,236.90,230.14,235.99,30127005.0
2021-03-24 16:00:00,237.85,238.00,235.32,235.46,25620127.0
2021-03-26 16:00:00,231.55,236.71,231.55,236.48,25479853.0


In [8]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(df['Volume'], order=(0, 1,1))
model_fit = model.fit()
print(model_fit.summary())

                               SARIMAX Results                                
Dep. Variable:                 Volume   No. Observations:                 1096
Model:                 ARIMA(0, 1, 1)   Log Likelihood              -13366.982
Date:                Sat, 26 Apr 2025   AIC                          26737.964
Time:                        14:41:12   BIC                          26747.961
Sample:                    04-01-2015   HQIC                         26741.747
                         - 03-30-2021                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.7530      0.025    -29.926      0.000      -0.802      -0.704
sigma2      1.732e+14   4.09e-17   4.24e+30      0.000    1.73e+14    1.73e+14
Ljung-Box (L1) (Q):                  14.17   Jarque-

In [9]:
import plotly.graph_objects as go
predictions = model_fit.predict()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Volume'], mode='lines', name='Actual'))
fig.add_trace(go.Scatter(x=df.index, y=predictions, mode='lines', name='Predicted'))

fig.update_layout(
    title='ARIMA Model Predictions',
    xaxis_title='Date',
    yaxis_title='Volume',
    legend_title='Data',
    hovermode="x unified"
)

fig.show()

In [10]:
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(df['Close'], order=(0, 1, 1))
model_fit = model.fit()
print(model_fit.summary())

                               SARIMAX Results                                
Dep. Variable:                  Close   No. Observations:                 1096
Model:                 ARIMA(0, 1, 1)   Log Likelihood               -1822.993
Date:                Sat, 26 Apr 2025   AIC                           3649.986
Time:                        14:41:13   BIC                           3659.983
Sample:                    04-01-2015   HQIC                          3653.769
                         - 03-30-2021                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1         -0.1274      0.023     -5.531      0.000      -0.173      -0.082
sigma2         5.6951      0.156     36.446      0.000       5.389       6.001
Ljung-Box (L1) (Q):                   2.21   Jarque-

In [11]:
import plotly.graph_objects as go
predictions = model_fit.predict()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df['Close'], mode='lines', name='Actual'))
fig.add_trace(go.Scatter(x=df.index, y=predictions, mode='lines', name='Predicted'))

fig.update_layout(
    title='ARIMA Model Predictions',
    xaxis_title='Date',
    yaxis_title='Close Price',
    legend_title='Data',
    hovermode="x unified"
)

fig.show()